# 🔬 Notebook 03 — Advanced Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 12, 'figure.dpi': 150})
sns.set_style('whitegrid')
closed = pd.read_csv('../data/closed_trades.csv')
merged = pd.read_csv('../data/merged_all_trades.csv')
S_ORDER = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
S_ORDER = [s for s in S_ORDER if s in closed['sentiment_class'].unique()]
S_PAL = {'Extreme Fear':'#d32f2f', 'Fear':'#f44336', 'Neutral':'#ff9800', 'Greed':'#4caf50', 'Extreme Greed':'#388e3c'}

## 10 Trader Segmentation (KMeans & PCA)

In [ ]:
account_features = closed.groupby('Account').agg(
    avg_pnl=('Closed PnL','mean'),
    win_rate=('Is Profitable','mean'),
    total_trades=('Closed PnL','count'),
    avg_size=('Size USD','mean'),
    long_ratio=('Trade Type', lambda x: (x=='Long').mean())
).reset_index()

X = StandardScaler().fit_transform(account_features.drop('Account',axis=1))
kmeans = KMeans(n_clusters=3, random_state=42)
account_features['Cluster'] = kmeans.fit_predict(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
account_features['PCA1'] = X_pca[:,0]
account_features['PCA2'] = X_pca[:,1]

plt.figure(figsize=(8,6))
sns.scatterplot(data=account_features, x='PCA1', y='PCA2', hue='Cluster', palette='Set1', s=100)
plt.title('Trader Clusters (PCA Projection)')
plt.savefig('../outputs/trader_clusters_pca.png')
plt.show()

print(account_features.groupby('Cluster').mean(numeric_only=True).drop(columns=['PCA1', 'PCA2']))

## Top Trader Dominance

In [ ]:
top_acct = account_features.sort_values('total_trades', ascending=False).iloc[0]
print(f"Top account has {top_acct['total_trades']:,} trades out of {len(closed):,} ({top_acct['total_trades']/len(closed)*100:.1f}%)")

## Contrarian Strategy Test (Long in Extreme Fear vs Short)

In [ ]:
ex_fear = closed[closed['sentiment_class'] == 'Extreme Fear']
if len(ex_fear) > 0:
    long_pnl = ex_fear[ex_fear['Trade Type'] == 'Long']['Closed PnL'].mean()
    short_pnl = ex_fear[ex_fear['Trade Type'] == 'Short']['Closed PnL'].mean()
    print(f'Extreme Fear - Avg Long PnL: ${long_pnl:.2f}')
    print(f'Extreme Fear - Avg Short PnL: ${short_pnl:.2f}')
else:
    print('Not enough Extreme Fear data')

## Lag Sentiment Analysis

In [ ]:
daily = closed.groupby('date').agg(Closed_PnL=('Closed PnL','mean'), lag1=('lag1','first'), lag3=('lag3','first'), lag7=('lag7','first')).dropna()
print("Spearman Correlation with Avg Daily PnL:")
for l in ['lag1', 'lag3', 'lag7']:
    r, p = spearmanr(daily[l], daily['Closed_PnL'])
    print(f"{l}: r={r:.3f}, p={p:.3f}")

## Liquidation Zone Analysis

In [ ]:
liqs = merged[merged['Direction'].str.contains('Liquidat', na=False) | merged['Direction'].str.contains('Auto-Delev', na=False)]
print("Liquidation Events:")
print(liqs[['date', 'sentiment_class', 'Size USD', 'Direction']])

## Coin Sensitivity Analysis

In [ ]:
top_coins = closed['Coin'].value_counts().head(5).index
coin_pnl = closed[closed['Coin'].isin(top_coins)].groupby(['Coin', 'sentiment_class'])['Closed PnL'].mean().unstack().reindex(columns=S_ORDER)
coin_pnl.plot(kind='bar', figsize=(10,5), color=[S_PAL[s] for s in S_ORDER])
plt.title('Coin Sensitivity to Sentiment (Avg PnL)')
plt.ylabel('Avg PnL')
plt.savefig('../outputs/coin_sensitivity.png')
plt.show()